# Conformal sensor escalation under operating-point shift — Paderborn replication and factor attribution

**Scope.** This notebook repeats, on the Paderborn KAt bearing dataset, every experiment of `01_kaist_conformal_gate.ipynb` with the same code (`run_experiments.py --dataset paderborn`; only the dataset adapter changes), and adds three analyses that the KAIST data cannot support: validation grouped by physical bearing rather than by recording (X1); attribution of the coverage collapse to a single operating factor, per sensing modality (X2); and transfer from artificially damaged to naturally damaged bearings (X3), produced by `run_paderborn_extra.py`.

**Data.** Paderborn KAt bearing data centre (Lessmeier et al., PHM Society European Conference, 2016): 32 bearings — 6 healthy, 12 with machined (artificial) damage and 14 with damage from accelerated-lifetime tests (real) — recorded at four operating points that differ from the baseline N15_M07_F10 (1500 rpm, 0.7 N·m, 1000 N) in exactly one factor: N09_M07_F10 (speed 900 rpm), N15_M01_F10 (torque 0.1 N·m) and N15_M07_F04 (radial force 400 N). One vibration channel and two motor-current phases are sampled at 64 kHz, plus a bearing-module temperature. Classes are healthy, outer-race damage, inner-race damage and combined damage (the last with only three bearings).

**Harmonisation with KAIST.** Vibration and current are resampled from 64 kHz to 25.6 kHz with an exact 2/5 polyphase filter; windows of 4096 samples with hop 2048 and the same `tfeat`/`ffeat` functions are used, so operation counts are identical across datasets (e.g. one accelerometer with time-domain features costs 36 864 operations in both). This analysis uses the *quick pass*: 5 of the 20 measurements per (bearing, operating point), i.e. 639 recordings and 31 266 windows (one recording failed to parse). The learning-curve result in Section 2 indicates that the remaining measurements would not change the ordering of configurations.

**Reproduction.** `python run_experiments.py --dataset paderborn` (E1–E6, about 45 min) and `python run_paderborn_extra.py` (X1–X3, about 1 h) write `results_paderborn/*.csv`; this notebook only reads them. `paderborn_lib.py` is the dataset adapter over the shared `cmlib.py`.

In [ ]:
# Environment setup: runs unchanged locally (repository root as working directory) and on Google Colab.
import os, sys, subprocess
REPO_URL = "https://github.com/candidoalfilho/conformal_escalation"   # update if the repository moves
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("cmlib.py"):                       # not inside the repository: fetch it
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "conformal_escalation"], check=True)
    os.chdir("conformal_escalation")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "scikit-learn", "matplotlib", "pandas", "scipy"], check=False)
print("working directory:", os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import report_lib as R
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 3)

import os
P  = pd.read_csv("results_paderborn/pareto.csv")
LC = pd.read_csv("results_paderborn/learning_curve.csv")
CC = pd.read_csv("results_paderborn/conformal_coverage.csv")
G  = pd.read_csv("results_paderborn/gate.csv")
RC = pd.read_csv("results_paderborn/recalibration.csv")
FC = pd.read_csv("results_paderborn/factor_coverage.csv")
DT = pd.read_csv("results_paderborn/damage_transfer.csv")
BP = pd.read_csv("results_paderborn/bearing_pareto.csv") if os.path.exists("results_paderborn/bearing_pareto.csv") else None
BC = pd.read_csv("results_paderborn/bearing_coverage.csv") if os.path.exists("results_paderborn/bearing_coverage.csv") else None
df = pd.read_parquet("data/paderborn_features.parquet")
OPS = {"N15_M07_F10": "baseline (1500 rpm, 0.7 N·m, 1000 N)", "N09_M07_F10": "speed 900 rpm", "N15_M01_F10": "torque 0.1 N·m", "N15_M07_F04": "radial force 400 N"}
print(f"{len(df):,} windows · {df.recording_id.nunique()} recordings · {df.bearing.nunique()} bearings · {df['class'].nunique()} classes")
display(df.groupby(["class", "damage_origin"]).bearing.nunique().rename("bearings").to_frame().T)
display(df.groupby(["class", "load"]).recording_id.nunique().unstack().rename(columns=OPS).rename_axis(columns="recordings per operating point"))

## Shared methodology

**Windowing and features.** Signals are segmented into windows of 4096 samples with a hop of 2048 (160 ms at 25.6 kHz), at most 50 windows per recording. For every channel, nine time-domain statistics are computed (mean, standard deviation, RMS, peak-to-peak, kurtosis, skewness, crest factor, shape factor, impulse factor) and, when the configuration includes the FFT, six spectral descriptors (dominant frequency, spectral centroid, spectral entropy, and band energies in 0–2 kHz, 2–6 kHz and 6–12.8 kHz). The temperature channel, which is quasi-constant within a window, contributes its mean and standard deviation. The same two functions (`tfeat`, `ffeat` in `extract_kaist.py` / `extract_paderborn.py`) are used for both datasets, so that feature vectors and costs are directly comparable.

**Sensing configurations.** A configuration is a set of channels and the choice of including the FFT. Its *cost in operations per window* is computed analytically (`kaist_lib.cost_ops`): 9 accumulations per sample for the time-domain statistics, $2.5\,N\log_2 N$ operations for a real radix-2 FFT of length $N$, plus magnitude and band accumulation per bin. The operation count isolates the arithmetic burden of each configuration on a microcontroller; it does not include acquisition energy, which is modelled separately (Section *Energy model*).

**Classifiers.** Three families are used so that any ordering of configurations is not an artefact of one learner: $\ell_2$-regularised logistic regression, a random forest (300 trees, minimum leaf size 2) and a multilayer perceptron (64–32 hidden units, tol $10^{-5}$, up to 3000 iterations). Convergence is checked for every fit and reported (`converged = 1.00` throughout).

**Validation protocols.** Windows from the same recording are strongly dependent (same mounting, thermal state, residual unbalance). All splits are therefore grouped by recording. Two protocols are used: *recording-grouped stratified 3-fold cross-validation*, repeated over three seeds, which estimates in-distribution performance; and *leave-one-operating-point-out* (LOLO), which trains on three operating points and tests on the fourth, which estimates performance under operating-point shift. Confidence intervals are 95 % percentile intervals from a cluster bootstrap that resamples *recordings* (600 replicates); a window-level bootstrap would understate uncertainty by roughly an order of magnitude.

**Split conformal prediction.** From the training recordings of each fold, a stratified third is carved out as a calibration set (never used for fitting). With non-conformity score $s_i = 1 - \hat p(y_i \mid x_i)$ (LAC) or the cumulative sorted probability up to the true class (APS), the conformal quantile $\hat q$ is the $\lceil (n+1)(1-\alpha)\rceil$-th smallest calibration score (`cmlib.conformal_qhat`; $\hat q = 1$ when that rank exceeds $n$) and the prediction set is $C(x) = \{y : s(x, y) \le \hat q\}$. Under exchangeability of calibration and test data, $\mathbb P\,(Y \in C(X)) \ge 1 - \alpha$ for any classifier. Two qualifications apply. First, windows of one recording are dependent, so exchangeability holds at recording level and the finite-sample statement is approximate, with an effective calibration size equal to the number of calibration recordings (hierarchical conformal prediction; Dunn, Wasserman & Ramdas, *JASA* 2023; Lee, Barber & Willett, 2023). Second, in the coverage and gate experiments (E3, E4, X1) the calibration scores of the three folds are pooled and a single $\hat q$ is applied to the pooled test predictions; the recalibration experiments (E5, E6, X3) use one quantile per held-out operating point. Coverage is reported as the empirical fraction of test windows whose true label lies in $C(x)$; the *gap* is coverage minus the nominal $1-\alpha$. In weighted conformal prediction (E6) calibration score $i$ carries a weight $w_i$, the test point weight 1, and $\hat q$ is the $(1-\alpha)$-quantile of the weighted empirical distribution normalised by $\sum_i w_i + 1$ (`cmlib.weighted_conformal_qhat`); with $w_i \equiv 1$ it reduces to the unweighted quantile, and for fixed weights the applicable guarantee is the total-variation bound of Barber et al. (2023).

**The escalation gate.** The cheap configuration is always on. A window is *escalated* to the expensive configuration if and only if its conformal set is not a singleton. The risk level $\alpha$ is the only parameter; the escalation fraction is measured, not tuned. The baseline *threshold gate* escalates when the top-class score $\max_y \hat p(y \mid x)$ falls below a swept threshold $\tau$. With the LAC score, $C(x)$ is a singleton iff the top-class probability is at least $1-\hat q$ and the second-highest is below it; when $1-\hat q > 0.5$ this coincides with a threshold on the top-class score, but at the quantiles observed here ($\hat q$ mostly above 0.5) the two gates are different selection rules, and the comparison is empirical. They are compared *at matched escalation fraction*: for every conformal operating point the threshold-gate curve is linearly interpolated at the same escalation fraction and the difference in macro-F1 is reported.

**Recalibration under shift.** For each held-out operating point, the model is trained on the remaining points and three calibration sets are compared: the *source* calibration recordings only; the source set *pooled* with $k$ labelled recordings from the target point; and the $k$ target recordings *only*. Test recordings are always disjoint from calibration recordings. Results are averaged over five seeds (random choice of the $k$ recordings).

**Dataset-specific protocol note.** In the Paderborn data the same physical bearing is recorded at all four operating points and five times per point. Grouping by *recording* therefore allows a bearing to appear in both training and test folds (at different operating points or measurements), which is optimistic for in-distribution accuracy. Section 7 repeats the main experiments with folds grouped by *bearing* to bound that optimism. The shift protocol (LOLO) is unaffected: each test operating point is entirely unseen.

In [ ]:
import paderborn_lib as K
blocks = K.feature_blocks(df)
display(pd.DataFrame([(R.PB_BLOCKS[b][0], R.PB_SHORT[b], len(cols), int(K.cost_ops(spec))) for b, (cols, spec) in blocks.items()],
                     columns=["configuration", "abbrev.", "n_features", "ops_per_window"]).set_index("configuration"))

## 1. Cost–accuracy frontier (E1)

Same encoding as in the KAIST notebook; the temperature-only configuration is omitted from the plot and reported in the table.

In [ ]:
R.fig_pareto(P, R.PB_BLOCKS, R.PB_SHORT, name="pb_fig1_pareto");
display(R.tab_pareto(P, R.PB_BLOCKS))

**The KAIST ordering replicates.** Vibration degrades under operating-point shift (one accelerometer with FFT: 0.959 → 0.762); current collapses (two phases with FFT: 0.908 → 0.612 with the random forest, 0.477 with the MLP); fusing accelerometer and current reaches 0.991 in distribution but 0.768 / 0.560 under shift, i.e. additional sensors do not help where it matters; temperature is again a confounder (0.374 → 0.288).

**What differs from KAIST, and matters.** With two current phases sampled at 64 kHz the current-only configuration reaches 0.908 in distribution — nearly on par with vibration — whereas on KAIST it never exceeded 0.42. Motor current therefore *does* carry the diagnosis as long as the operating point is the one it was trained and calibrated on; the question of when it can be trusted is exactly the one the conformal gate is meant to answer, and Section 3 shows that this is where the guarantee fails. Two caveats apply to the in-distribution figures. First, recording-grouped folds let the same physical bearing appear in training and test; Section 7 shows that bearing-grouped validation lowers every in-distribution macro-F1 to 0.4–0.55, so the recording-grouped numbers measure recognition of known bearings at known operating points, not generalisation to new bearings. The shift-protocol numbers, which carry the central claim, are not affected in their interpretation: because the same bearings appear on both sides of the leave-one-operating-point-out split, the degradation they show is attributable to the operating point alone. Second, the two-phase current configuration costs 335 872 operations per window, twice the single-accelerometer configuration with FFT (167 936): current is not the cheap tier even in operations, and in energy the ratio is reversed by a further order of magnitude (energy model in the companion notebook). The only cascade with a cheap tier worth the name is therefore *one accelerometer, time-domain → one accelerometer, time+FFT* (4.6× in operations).

## 2. Learning curve in windows per recording (E2)

In [ ]:
R.fig_learning_curve(LC, R.PB_BLOCKS, name="pb_fig2_learning_curve");
display(LC.pivot_table(index="n_windows", columns=["block", "model"], values="f1_mean").round(3))

Saturated, as on KAIST (one accelerometer with FFT, random forest: 0.950 at 10 windows per recording, 0.959 at 50). The quick pass with 5 of 20 measurements per setting is therefore sufficient for the ordering of configurations; the full pass would narrow intervals, not change conclusions.

## 3. Empirical coverage of split conformal prediction (E3)

The cheap tier is now the two-phase current configuration (the direct analogue of the KAIST cascade). Each held-out operating point in panel (b) differs from the baseline in one factor, which is what allows the attribution in Section 8.

In [ ]:
R.fig_coverage(CC, per_group_label=lambda c: OPS[c.replace("cov_", "")].replace("baseline ", "baseline\n"), name="pb_fig3_coverage");
display(R.tab_coverage_gap(CC))
sel = CC[(CC.protocol=="lolo") & (CC.score=="lac") & np.isclose(CC.alpha, 0.20)]
display(sel.groupby("model")[[c for c in CC.columns if c.startswith("cov_")] + ["coverage", "avg_size", "empty_frac"]].mean().rename(columns=lambda c: OPS.get(c.replace("cov_",""), c)).round(3).T)
print("coverage per class at nominal 80 % (LAC), mean over classifiers and seeds; classes: 0 healthy, 1 outer race, 2 inner race, 3 combined")
sel = CC[(CC.score=="lac") & np.isclose(CC.alpha, 0.20)]
display(sel.groupby("protocol")[[c for c in CC.columns if c.startswith("covclass_")]].mean().round(3))

**The collapse replicates and is stronger.** Under recording-grouped validation the LAC gap is −0.009 on average (worst −0.039); under leave-one-operating-point-out it is **−0.296 on average and −0.501 in the worst case**, against −0.148 / −0.453 on KAIST. At nominal 80 % the gate delivers 50.0 % (logistic regression) and 32.6 % (random forest). APS stays at or above nominal by producing very large sets (99.8 % coverage with the random forest under shift, mean set size 3.5 of 4 classes), which hides rather than solves the problem. Per class, shift lowers the coverage of every fault class to 0.34–0.41 while the healthy class keeps 0.57: the gate's failure is concentrated on the missed-detection side.

**Per held-out operating point** (LAC, α = 0.20): changing the rotational speed gives 0.33 / 0.01 coverage (logistic regression / random forest) and changing the torque 0.31 / 0.11, while changing the radial force gives 0.69 / 0.65. For the current-based cheap tier, speed and torque destroy exchangeability and radial force barely affects it. Section 8 shows that this attribution is specific to the modality.

**A note on the random forest.** Under shift 62 % of its LAC sets are *empty*: the calibrated quantile is small because in-distribution probabilities are confident, and under shift no class clears it. An empty set is not a singleton, so the gate does escalate; but the phenomenon is a symptom of over-confidence that the logistic regression does not show, and it is reported rather than hidden.

## 4. Conformal gate versus threshold gate (E4)

In [ ]:
CASC = {"cur2_full->vib1_full": "2 current phases (time+FFT) → 1 accelerometer (time+FFT)",
        "vib1_time->vib1_full": "1 accelerometer (time) → 1 accelerometer (time+FFT)"}
R.fig_gate(G, CASC, model="rf", name="pb_fig4_gate");
display(R.tab_gate_delta(G, protocol="grouped"))
d = G[(G.model=="rf") & (G.gate=="conformal_lac") & (G.cascade=="vib1_time->vib1_full")]
display(d.groupby(["protocol", "knob"])[["esc_frac", "f1", "avg_cost_ops"]].mean().round(3))

**Same curve, again.** At matched escalation fraction the conformal gate and the threshold gate are indistinguishable (table above). For the accelerometer-first cascade with the random forest in distribution, α = 0.05 escalates 21.9 % of windows and reaches a macro-F1 of 0.925 at a mean cost of 73.7 k operations, against 0.941 at 167.9 k for the always-on expensive tier (fitted on the same calibration split; the full-training figure in Section 1 is 0.959). Empty sets count as escalation, as the gate definition requires.

**Under shift the gate under-escalates.** With leave-one-operating-point-out validation and α = 0.10, the LAC gate with the random forest escalates only 6 % of windows (against 6 % in distribution) and delivers a macro-F1 of 0.719 instead of 0.890 (the always-on expensive tier gives 0.753 under the same shift): the sets are singletons *and wrong*. A gate whose guarantee has silently expired does not ask for the expensive sensor precisely when it should. On the current tier the same classifier fails the other way, with 62 % empty sets and therefore over-escalation (Section 3); which mode a deployment gets depends on how the cheap model behaves off-distribution and is not observable by the node. This is the safety-relevant reading of the coverage collapse, and the reason recalibration at the deployment point (Section 5) is a requirement rather than a refinement.

## 5. Recalibration at the target operating point (E5)

In [ ]:
R.fig_recal(RC, group_label=lambda s: OPS[s], name="pb_fig5_recalibration");
print("gap by k and calibration mix, mean over both classifiers and α ∈ {0.05, 0.10, 0.20, 0.30}"); display(R.tab_recal(RC))
print("per classifier at α = 0.20"); display(pd.concat({m: R.tab_recal(RC, model=m, alpha=0.20) for m in ["logreg", "rf"]}, axis=1))

**Three to five target recordings restore nominal coverage.** Averaged over both classifiers and the four risk levels, the gap moves from −0.337 with source calibration only to −0.037 at $k=3$ and +0.008 at $k=5$. At α = 0.20 the random forest goes from −0.458 to −0.018 ($k=3$) and +0.006 ($k=5$), the logistic regression from −0.299 to −0.017 and +0.064.

**Pooling with the source calibration set is nearly useless here.** The source set contains about 31 000 windows; $k$ target recordings add a few hundred. The pooled quantile barely moves (−0.337 → −0.311 at $k=8$ in the aggregate; −0.458 → −0.435 for the random forest at α = 0.20). On KAIST pooling was worse than replacing; on Paderborn it is close to doing nothing, because the stale scores dominate by volume. The conclusion is the same and sharper: the calibration set must be *exchangeable* with the deployment data, and volume does not compensate. Per held-out operating point (target-only calibration), the speed and torque shifts are repaired with $k = 2$–$3$ (gaps +0.01 and −0.09 at $k = 2$, +0.08 and +0.06 at $k = 3$), the radial-force shift needs $k = 5$ (−0.24 at $k = 3$, −0.01 at $k = 5$).

## 5b. Weighted conformal prediction (E6)

As in the KAIST notebook: source scores keep a fixed weight $\rho$ in the pooled calibration set ($\rho = 1$ pooled, $\rho = 0$ target only). With about 31 000 source windows against a few hundred target windows, the question is whether any fixed weight short of zero can neutralise the stale set.

In [ ]:
W = pd.read_csv("results_paderborn/weighted_recalibration.csv") if os.path.exists("results_paderborn/weighted_recalibration.csv") else None
if W is not None:
    R.fig_weighted(W, name="pb_fig5b_weighted");
    g, s = R.tab_weighted(W); print("coverage gap"); display(g); print("mean set size"); display(s)
else:
    print("weighted_recalibration.csv not found — run `python run_experiments.py --dataset paderborn e6`")

With a source set two orders of magnitude larger than the target set, a weight of $\rho = 0.3$ still leaves the source scores in control of the quantile; only $\rho \le 0.03$ approaches target-only calibration. What governs the outcome is the target share of the normalised weight, $n_{\mathrm{tgt}} / (n_{\mathrm{tgt}} + \rho\, n_{\mathrm{src}})$: with $n_{\mathrm{src}} \approx 7\,700$ windows and a few hundred target windows, $\rho = 0.3$ leaves that share below 0.1. Likelihood-ratio weights (Tibshirani et al., 2019) are density ratios, not functions of the set sizes, and were not evaluated. For a commissioning rule the simple statement stands: once three target recordings exist, calibrate on them.

## 6. Summary of the replication

| quantity | KAIST | Paderborn |
|---|---|---|
| mean LAC gap, recording-grouped | +0.045 | −0.009 |
| mean LAC gap, bearing-grouped | — | +0.002 to +0.014 |
| mean LAC gap, operating-point shift | −0.148 | −0.296 |
| worst LAC gap, shift | −0.453 | −0.501 |
| coverage delivered at nominal 80 %, shift | 55.3 % | 50.0 % (LR) / 32.6 % (RF) |
| Δ macro-F1 conformal − threshold gate, matched escalation | −0.000 ± 0.004 | −0.002 ± 0.004 |
| target recordings to restore coverage (mean over classifiers and α) | 5 | 3–5 |
| pooled vs replaced calibration at $k=3$ (same aggregation) | −0.100 vs −0.052 | −0.327 vs −0.037 |

Two independent benchmarks, different machines (a motor-driven shaft with four accelerometers; a bearing test rig with two current phases), different classes, the same pipeline and the same conclusions.

## 7. Validation grouped by bearing (X1)

Folds are grouped by physical bearing (stratified 3-fold, three seeds; the combined-damage class has only three bearings, which caps the number of folds). Calibration recordings for the coverage experiment are also carved by bearing.

In [ ]:
if BP is not None:
    R.fig_protocol_compare(P, BP, R.PB_BLOCKS, R.PB_SHORT, name="pb_fig6_bearing_grouped");
    t = R.tab_pareto(P, R.PB_BLOCKS)[["grouped_F1", "lolo_F1"]].join(
        R.tab_pareto(BP, R.PB_BLOCKS, protocols=("bearing",))[["bearing_F1", "bearing_CI95"]])
    display(t)
    bc = BC.copy(); bc["gap"] = bc.coverage - bc.nominal
    print("coverage gap with bearing-grouped folds and bearing-carved calibration (cheap tiers)")
    display(bc.groupby(["block", "score"])["gap"].agg(mean="mean", worst="min").round(3))
else:
    print("bearing_pareto.csv not found — run `python run_paderborn_extra.py x1`")

**Recording-grouped validation overstates in-distribution accuracy by a wide margin on this dataset.** With folds grouped by bearing, the single-accelerometer configuration with FFT falls from 0.959 to 0.553 (random forest, 95 % CI [0.40, 0.67]), the two-phase current configuration from 0.908 to 0.458, and the fusion configuration from 0.991 to 0.504. Hand-crafted statistics with shallow classifiers recognise *bearings* far better than they recognise *damage classes* on unseen bearings — a leakage mode documented for this benchmark (Vieira et al., 2025; Kaya & Jobani, 2026). The bearing-grouped figures are the honest in-distribution reference for the Paderborn data and are the ones to compare with the literature.

**The conformal guarantee is unaffected by the weakness of the classifier.** With bearing-grouped folds and bearing-carved calibration, the LAC coverage gap is +0.002 (current) and +0.012–0.014 (vibration): coverage holds whenever calibration and test are exchangeable, whatever the accuracy, which is the property the gate relies on. The guarantee fails under operating-point shift (Section 3), not under bearing novelty.

**What this changes in the reading of the shift results.** The leave-one-operating-point-out split shares bearings between training and test, so the coverage collapse it exposes is attributable to the operating point alone and not confounded with unseen bearings; the same holds for the factor attribution of Section 8. The in-distribution comparison between current and vibration (Section 1), on the other hand, must be read as bearing-level recognition; on unseen bearings the two modalities are closer (0.46–0.53 vs 0.55) and both are modest.

## 8. Which factor breaks exchangeability, per sensing modality (X2)

Models are fitted and calibrated on recordings of the baseline operating point only (fit / calibration / in-domain test split by recording) and evaluated on each neighbouring operating point. Because each neighbour changes one factor, the loss of coverage can be attributed to that factor. The experiment is run for the current-only tier, the single-accelerometer time-domain tier and the full vibration tier.

In [ ]:
R.fig_factor_modality(FC, R.PB_BLOCKS, alpha=0.20, score="lac", name="pb_fig7_factor_modality");
s = FC[(FC.score=="lac") & np.isclose(FC.alpha, 0.20)]
print("empirical coverage at nominal 80 % (LAC)"); display(s.pivot_table(index=["block","model"], columns="factor", values="coverage")[R.FACTOR_ORDER].rename(columns=R.FACTOR_LABEL).round(3))
print("macro-F1 of the same models on the same targets"); display(s.pivot_table(index=["block","model"], columns="factor", values="f1")[R.FACTOR_ORDER].rename(columns=R.FACTOR_LABEL).round(3))

**The attribution is modality-specific.** For the vibration tiers, a change of **rotational speed** destroys coverage (0.07–0.50 at nominal 80 %) while a change of **torque leaves it essentially intact** (0.67–0.82, with the logistic regression at nominal) and a change of radial force degrades it moderately (0.27–0.73). For the current tier, both speed and torque destroy coverage (0.01–0.38). Macro-F1 on the same targets tells the same story (vibration under torque shift: 0.64–0.90; under speed shift: 0.22–0.31).

This reconciles the result with accuracy-based studies of domain shift on Paderborn, which report that rotational speed drives the degradation while load and radial force have small effects (Panić et al., *Reliability Engineering & System Safety*, 2027; Alam et al., 2025): those studies use vibration. The torque sensitivity observed in Section 3 belongs to the *current-based* cheap tier, as the KAIST load-shift results (vibration 0.69 → 0.69, current 0.42 → 0.21) independently confirm. The physics is consistent with the attribution. Speed scales the impact rate and the vibration amplitude, so every time-domain statistic moves (which is why the time-domain-only vibration tier collapses as well), and on an inverter-fed motor it moves the current fundamental and every fault sideband. A torque change from 0.7 to 0.1 N·m is a roughly seven-fold change in q-axis current: every amplitude statistic of the current channel rescales while the fault sidebands stay small relative to the fundamental, the classical load dependence of motor-current signature analysis, which the gate inherits; the vibration signature is mainly unaffected. Radial force changes the load zone and hence the amplitude of outer-race impulses. Whether speed-normalised (order-tracked) features would remove the speed sensitivity is an untested control.

The consequence for cascade design is direct. A current-first cascade must be recalibrated whenever speed *or* torque changes; an accelerometer-first cascade when speed changes and, to a lesser degree, when radial force changes. Which operating factors a plant varies is therefore part of the sensor-selection decision, together with energy.

## 9. Transfer from artificial to real damage (X3)

Models are trained on bearings with machined damage (plus three healthy bearings) and evaluated on bearings with damage from accelerated-lifetime tests (plus the other three healthy bearings), and vice versa. The machined-damage set has no combined-damage class, so the artificial→real direction tests a three-class model on four classes and its macro-F1 is bounded by 0.75 by construction. Calibration is carved by bearing from the training side; recalibration uses $k$ recordings of the test side, as in Section 5.

In [ ]:
s = DT[(DT.score=="lac") & np.isclose(DT.alpha, 0.20)]
print("macro-F1 of source-trained models on the target damage origin (chance level for 4 classes ≈ 0.25)")
display(s.groupby(["train","test","block","model"]).f1_target.mean().unstack().round(3))
R.fig_damage_transfer(DT, {b: R.PB_BLOCKS[b] for b in ["cur2_full","vib1_time","vib1_full"]}, model="rf", name="pb_fig8_damage_transfer");
t = s[s.model=="rf"].assign(gap=lambda d: d.coverage-d.nominal)
display(t.pivot_table(index=["train","block","k"], columns="mix", values=["gap","avg_size"]).round(3))

**Hand-crafted features do not transfer between damage origins.** Macro-F1 on the unseen damage origin is 0.19–0.29 for every tier and classifier, i.e. chance level for four classes, in both directions. This is a failure of the classifier, not only of its calibration, and it is consistent with the known difficulty of the artificial-to-real split on this benchmark.

**Recalibration cannot rescue an uninformative classifier.** With source calibration the gap is −0.37 (current) and −0.13 to −0.17 (vibration); target-only recalibration moves coverage towards nominal, but erratically across $k$ and only by inflating the sets (see `avg_size`): when the classifier is at chance, the only set with 80 % coverage is close to the full label set, and a gate built on it would escalate every window. Operating-point shift (Sections 3–5 and 8) is a *calibration* problem that a few labelled recordings fix; damage-origin shift is a *representation* problem that they do not. The distinction bounds the claim of this work: recalibration at the deployment point restores the guarantee when the model remains informative there.

## 10. Limitations and open items

* The quick pass uses 5 of 20 measurements per setting; intervals would narrow with the full pass but the learning curve indicates no change in ordering.
* The combined-damage class has three bearings; macro-F1 is sensitive to it.
* Recording-grouped in-distribution numbers are strongly optimistic for this dataset (bearing identity leaks); bearing-grouped numbers (Section 7) are the honest reference and should be the ones reported for accuracy.
* Energy figures are datasheet-derived; the FFT cycle count on the ESP32-S3 is extrapolated and remains to be measured on hardware.
* Weighted conformal prediction was evaluated only with fixed source weights (Section 5b); likelihood-ratio weighting and online (adaptive) conformal inference remain to be compared.
* The 3–5 recordings of the commissioning rule must be *labelled by class*; on a production machine only healthy recordings exist at a new operating point, which recalibrate the healthy-class quantile alone (class-conditional / Mondrian calibration).
* No deep baseline on raw signals is included; the phenomena reported concern the calibration of a given classifier, but the magnitude of the transfer failures in Section 9 may differ for learned representations.